In [1]:
# =====================================================
# 05 Model Training - Rider-Level Customer Churn
# =====================================================

import pandas as pd
import numpy as np
import joblib

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score
)

sns.set_theme(style="whitegrid")

# =====================================================
# Load Rider-Level Feature Dataset
# =====================================================

df = pd.read_csv("../data/processed/rider_level_churn_dataset.csv")

print("Dataset shape:", df.shape)
display(df.head())

# =====================================================
# Define Target
# =====================================================

target = "is_churned"
y = df[target]

# =====================================================
# Drop Columns Not Needed for Modelling
# =====================================================

drop_cols = [
    "is_churned",
    "user_id",
    "signup_date",
    "referred_by"
]

X = df.drop(columns=drop_cols, errors="ignore")

print("Feature shape:", X.shape)
print("Target shape:", y.shape)

print("\nTarget distribution:")
print(y.value_counts(normalize=True) * 100)

# Confirm dropped columns are gone
assert all(col not in X.columns for col in drop_cols), "Some drop columns are still in X."
print("\nValidation passed: unwanted columns removed.")

# =====================================================
# Identify Numerical and Categorical Features
# =====================================================

categorical_features = X.select_dtypes(
    include=["object", "category", "string"]
).columns.tolist()

numerical_features = X.select_dtypes(
    include=["int64", "int32", "float64", "float32"]
).columns.tolist()

print("\nCategorical features:")
print(categorical_features)

print("\nNumerical features:")
print(numerical_features)

# =====================================================
# Train-Test Split
# =====================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

print("\nTrain target distribution:")
print(y_train.value_counts(normalize=True) * 100)

print("\nTest target distribution:")
print(y_test.value_counts(normalize=True) * 100)

# =====================================================
# Preprocessing Pipeline
# =====================================================

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

# =====================================================
# Evaluation Function
# =====================================================

def evaluate_model(model_name, model, X_test, y_test, threshold=0.50):
    
    probs = model.predict_proba(X_test)[:, 1]
    preds = (probs >= threshold).astype(int)

    accuracy = accuracy_score(y_test, preds)
    precision = precision_score(y_test, preds, zero_division=0)
    recall = recall_score(y_test, preds, zero_division=0)
    f1 = f1_score(y_test, preds, zero_division=0)
    roc_auc = roc_auc_score(y_test, probs)

    print("=" * 60)
    print(f"{model_name} | Threshold: {threshold}")
    print("=" * 60)

    print("\nConfusion Matrix:")
    print(confusion_matrix(y_test, preds))

    print("\nClassification Report:")
    print(classification_report(y_test, preds, zero_division=0))

    print("\nPerformance Metrics")
    print("-" * 40)
    print(f"Accuracy : {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall   : {recall:.4f}")
    print(f"F1-Score : {f1:.4f}")
    print(f"ROC-AUC  : {roc_auc:.4f}")

    return {
        "Model": model_name,
        "Threshold": threshold,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1,
        "ROC-AUC": roc_auc
    }

# =====================================================
# Build Models
# =====================================================

scale_pos_weight = y_train.value_counts()[0] / y_train.value_counts()[1]

log_reg_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            LogisticRegression(
                max_iter=3000,
                class_weight="balanced",
                random_state=42
            )
        )
    ]
)

rf_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            RandomForestClassifier(
                n_estimators=500,
                max_depth=10,
                min_samples_leaf=5,
                min_samples_split=10,
                class_weight="balanced",
                random_state=42,
                n_jobs=-1
            )
        )
    ]
)

xgb_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            XGBClassifier(
                n_estimators=500,
                max_depth=3,
                learning_rate=0.03,
                subsample=0.9,
                colsample_bytree=0.9,
                min_child_weight=5,
                gamma=1,
                scale_pos_weight=scale_pos_weight,
                objective="binary:logistic",
                eval_metric="logloss",
                random_state=42,
                n_jobs=-1
            )
        )
    ]
)

models = {
    "Logistic Regression": log_reg_model,
    "Random Forest": rf_model,
    "XGBoost": xgb_model
}

# =====================================================
# Cross-Validation
# =====================================================

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

cv_results = []

for name, model in models.items():
    scores = cross_val_score(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring="roc_auc",
        n_jobs=-1
    )

    cv_results.append({
        "Model": name,
        "Mean ROC-AUC": scores.mean(),
        "Std ROC-AUC": scores.std()
    })

cv_results_df = pd.DataFrame(cv_results)

print("\nCross-Validation ROC-AUC Results:")
display(cv_results_df.sort_values(by="Mean ROC-AUC", ascending=False))

# =====================================================
# Train and Evaluate Models at Default Threshold
# =====================================================

model_results = []

for name, model in models.items():
    model.fit(X_train, y_train)

    result = evaluate_model(
        name,
        model,
        X_test,
        y_test,
        threshold=0.50
    )

    model_results.append(result)

model_results_df = pd.DataFrame(model_results)

print("\nModel Comparison at Threshold 0.50:")
display(model_results_df.sort_values(by="ROC-AUC", ascending=False))

# =====================================================
# Threshold Tuning
# =====================================================

thresholds = np.arange(0.20, 0.61, 0.05)

threshold_results = []

for name, model in models.items():
    probs = model.predict_proba(X_test)[:, 1]

    for threshold in thresholds:
        preds = (probs >= threshold).astype(int)

        threshold_results.append({
            "Model": name,
            "Threshold": threshold,
            "Accuracy": accuracy_score(y_test, preds),
            "Precision": precision_score(y_test, preds, zero_division=0),
            "Recall": recall_score(y_test, preds, zero_division=0),
            "F1 Score": f1_score(y_test, preds, zero_division=0),
            "ROC-AUC": roc_auc_score(y_test, probs)
        })

threshold_results_df = pd.DataFrame(threshold_results)

print("\nThreshold Tuning Results:")
display(
    threshold_results_df.sort_values(
        by="F1 Score",
        ascending=False
    ).head(15)
)

# =====================================================
# Select Best Model Based on F1 Score
# =====================================================

best_row = (
    threshold_results_df
    .sort_values(by="F1 Score", ascending=False)
    .iloc[0]
)

best_model_name = best_row["Model"]
best_threshold = best_row["Threshold"]

best_model = models[best_model_name]

print("\nBest Model Selection")
print("-" * 40)
print("Best model:", best_model_name)
print("Best threshold:", best_threshold)

# Evaluate best model again using best threshold
best_results = evaluate_model(
    best_model_name,
    best_model,
    X_test,
    y_test,
    threshold=best_threshold
)

# =====================================================
# Feature Importance for XGBoost or Random Forest
# =====================================================

if best_model_name in ["XGBoost", "Random Forest"]:

    feature_names = best_model.named_steps["preprocessor"].get_feature_names_out()

    importances = best_model.named_steps["classifier"].feature_importances_

    feature_importance_df = (
        pd.DataFrame({
            "Feature": feature_names,
            "Importance": importances
        })
        .sort_values(by="Importance", ascending=False)
    )

    print("\nTop 20 Feature Importances:")
    display(feature_importance_df.head(20))

    plt.figure(figsize=(10, 8))

    sns.barplot(
        data=feature_importance_df.head(20),
        x="Importance",
        y="Feature",
        hue="Feature",
        legend=False
    )

    plt.title(f"Top 20 Feature Importances - {best_model_name}")
    plt.xlabel("Importance")
    plt.ylabel("Feature")

    plt.tight_layout()
    plt.show()

# =====================================================
# Save Best Model and Threshold
# =====================================================

joblib.dump(
    best_model,
    "../models/churn_prediction_model.pkl"
)

joblib.dump(
    best_threshold,
    "../models/churn_prediction_threshold.pkl"
)

print("\nBest model and threshold saved successfully.")
print(f"Saved model: {best_model_name}")
print(f"Saved threshold: {best_threshold}")

Dataset shape: (10000, 35)


,user_id,signup_date,loyalty_status,age,city,avg_rating_given,referred_by,is_referred,is_churned,total_trips,...,favourite_vehicle_type,total_sessions,avg_time_on_app,avg_pages_visited,conversion_rate,account_tenure_days,trip_velocity,trips_per_session,engagement_score,age_group
0,R00000,2025-01-24 00:00:00+00:00,Bronze,34.729629,Nairobi,5.0,R00001,1,0,25,...,Sedan,4.0,92.000000,3.000000,0.25,93,0.268817,6.250000,39.000000,26-35
1,R00001,2024-09-09 00:00:00+00:00,Bronze,34.571020,Nairobi,4.7,unknown,1,0,14,...,Sedan,3.0,174.666667,2.666667,0.00,230,0.060870,4.666667,71.600000,26-35
2,R00002,2024-09-07 00:00:00+00:00,Bronze,47.133960,Lagos,4.2,unknown,1,0,24,...,Sedan,3.0,191.000000,3.000000,0.00,232,0.103448,8.000000,78.200000,46-60
3,R00003,2025-03-17 00:00:00+00:00,Bronze,41.658628,Nairobi,4.9,unknown,1,1,9,...,Suv,3.0,75.333333,1.666667,0.00,41,0.219512,3.000000,31.666667,36-45
4,R00004,2024-08-20 00:00:00+00:00,Silver,40.681709,Lagos,3.9,R00002,1,0,16,...,Sedan,2.0,17.000000,2.500000,0.00,250,0.064000,8.000000,8.100000,36-45


Feature shape: (10000, 31)
Target shape: (10000,)

Target distribution:
is_churned
0    81.14
1    18.86
Name: proportion, dtype: float64

Validation passed: unwanted columns removed.

Categorical features:
['loyalty_status', 'city', 'favourite_payment_type', 'favourite_weather', 'favourite_vehicle_type', 'age_group']

Numerical features:
['age', 'avg_rating_given', 'is_referred', 'total_trips', 'avg_fare', 'total_fare', 'avg_surge', 'avg_tip', 'avg_tip_percentage', 'avg_trip_duration', 'avg_fare_per_minute', 'avg_driver_rating', 'avg_driver_acceptance', 'weekend_trip_rate', 'peak_hour_trip_rate', 'night_trip_rate', 'surge_trip_rate', 'total_sessions', 'avg_time_on_app', 'avg_pages_visited', 'conversion_rate', 'account_tenure_days', 'trip_velocity', 'trips_per_session', 'engagement_score']
X_train: (8000, 31)
X_test: (2000, 31)

Train target distribution:
is_churned
0    81.1375
1    18.8625
Name: proportion, dtype: float64

Test target distribution:
is_churned
0    81.15
1    18.85
Na

,Model,Mean ROC-AUC,Std ROC-AUC
0,Logistic Regression,0.608500,0.010877
1,Random Forest,0.600797,0.006605
2,XGBoost,0.583827,0.010100


Logistic Regression | Threshold: 0.5

Confusion Matrix:
[[940 683]
 [149 228]]

Classification Report:
              precision    recall  f1-score   support

           0       0.86      0.58      0.69      1623
           1       0.25      0.60      0.35       377

    accuracy                           0.58      2000
   macro avg       0.56      0.59      0.52      2000
weighted avg       0.75      0.58      0.63      2000


Performance Metrics
----------------------------------------
Accuracy : 0.5840
Precision: 0.2503
Recall   : 0.6048
F1-Score : 0.3540
ROC-AUC  : 0.6141
Random Forest | Threshold: 0.5

Confusion Matrix:
[[1319  304]
 [ 266  111]]

Classification Report:
              precision    recall  f1-score   support

           0       0.83      0.81      0.82      1623
           1       0.27      0.29      0.28       377

    accuracy                           0.71      2000
   macro avg       0.55      0.55      0.55      2000
weighted avg       0.73      0.71      0.72  

,Model,Threshold,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,Logistic Regression,0.5,0.584,0.250274,0.604775,0.354037,0.614082
1,Random Forest,0.5,0.715,0.267470,0.294430,0.280303,0.613636
2,XGBoost,0.5,0.625,0.249664,0.493369,0.331551,0.602486



Threshold Tuning Results:


,Model,Threshold,Accuracy,Precision,Recall,F1 Score,ROC-AUC
6,Logistic Regression,0.50,0.5840,0.250274,0.604775,0.354037,0.614082
13,Random Forest,0.40,0.5150,0.235504,0.700265,0.352470,0.613636
5,Logistic Regression,0.45,0.4755,0.227273,0.742706,0.348042,0.614082
23,XGBoost,0.45,0.5370,0.235804,0.649867,0.346045,0.602486
14,Random Forest,0.45,0.6200,0.254802,0.527851,0.343696,0.613636
22,XGBoost,0.40,0.4440,0.220957,0.771883,0.343566,0.602486
4,Logistic Regression,0.40,0.3820,0.213857,0.851459,0.341853,0.614082
12,Random Forest,0.35,0.4010,0.215128,0.822281,0.341034,0.613636
21,XGBoost,0.35,0.3575,0.207474,0.854111,0.333852,0.602486
24,XGBoost,0.50,0.6250,0.249664,0.493369,0.331551,0.602486



Best Model Selection
----------------------------------------
Best model: Logistic Regression
Best threshold: 0.49999999999999994
Logistic Regression | Threshold: 0.49999999999999994

Confusion Matrix:
[[940 683]
 [149 228]]

Classification Report:
              precision    recall  f1-score   support

           0       0.86      0.58      0.69      1623
           1       0.25      0.60      0.35       377

    accuracy                           0.58      2000
   macro avg       0.56      0.59      0.52      2000
weighted avg       0.75      0.58      0.63      2000


Performance Metrics
----------------------------------------
Accuracy : 0.5840
Precision: 0.2503
Recall   : 0.6048
F1-Score : 0.3540
ROC-AUC  : 0.6141

Best model and threshold saved successfully.
Saved model: Logistic Regression
Saved threshold: 0.49999999999999994


In [2]:
for test_size in [0.2, 0.3]:
    print(f"\nTesting {int((1-test_size)*100)}/{int(test_size*100)} split")

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=test_size,
        random_state=42,
        stratify=y
    )

    # Train your Logistic Regression
    log_reg_model.fit(X_train, y_train)

    # Train your Random Forest
    rf_model.fit(X_train, y_train)

    # Train your XGBoost
    xgb_model.fit(X_train, y_train)

    # Evaluate each model here


Testing 80/20 split

Testing 70/30 split


In [3]:
for test_size in [0.2, 0.3]:
    print(f"\nTesting {int((1-test_size)*100)}/{int(test_size*100)} split")

    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=test_size,
        random_state=42,
        stratify=y
    )

    log_reg_model.fit(X_train, y_train)


Testing 80/20 split

Testing 70/30 split


In [4]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

for test_size in [0.2, 0.3]:
    print(f"\n===== Testing {int((1-test_size)*100)}/{int(test_size*100)} Split =====")

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=test_size,
        random_state=42,
        stratify=y
    )

    rf_model.fit(X_train, y_train)

    preds = rf_model.predict(X_test)
    probs = rf_model.predict_proba(X_test)[:, 1]

    print("Accuracy :", accuracy_score(y_test, preds))
    print("Precision:", precision_score(y_test, preds))
    print("Recall   :", recall_score(y_test, preds))
    print("F1 Score :", f1_score(y_test, preds))
    print("ROC AUC  :", roc_auc_score(y_test, probs))


===== Testing 80/20 Split =====
Accuracy : 0.715
Precision: 0.2674698795180723
Recall   : 0.29442970822281167
F1 Score : 0.2803030303030303
ROC AUC  : 0.613635880765717

===== Testing 70/30 Split =====
Accuracy : 0.736
Precision: 0.2801556420233463
Recall   : 0.254416961130742
F1 Score : 0.26666666666666666
ROC AUC  : 0.6150783511560316


In [5]:
print(y.value_counts())
print(y.value_counts(normalize=True) * 100)

is_churned
0    8114
1    1886
Name: count, dtype: int64
is_churned
0    81.14
1    18.86
Name: proportion, dtype: float64
